# 01b · Prepare the AWE industrial dataset (6 clients, spike backdoor)
Download https://github.com/youngmanyd/NGIT-2022-AWE-Industrial-Dataset and place the folders
`0chips/ … 5chips/` inside `data/AWE/`.

Pipeline (unchanged from the repo): z-score → 4th-order low-pass (250 Hz) → downsample 4 kHz→500 Hz →
windows of 30 rows (stride 10) × 1000 features. Client *k* holds mostly eccentricity level *k* (groups 0-3) plus random
other groups. Attacker `client_2`: spike trigger (+20 on several positions, noise σ=0.5) on **all** its windows → class 5.

Fixes vs the repo: the config keys `SAVE_FOLDER / CLIENTS / SEED` did not exist (KeyError); trigger noise now uses a seeded RNG;
`num_clients` is 6 (the repo config said 10 while the data code hard-codes 6).
**Untested on the real files** (the dataset was not available while writing this) - the function raises a clear error if the window shape is not (30, 1000).

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        break
print("project root:", ROOT)

In [ ]:
from fuguard_repro import colab
colab.fetch_awe_dataset()      # git clone + link '<k>chips' folders into data/AWE (untested; follow its message if it can't find them)

In [ ]:
from fuguard_repro.config import awe_config
from fuguard_repro.data_awe import prepare_awe_data
prepare_awe_data(awe_config())